# MonoDETR R0 Vehicle + Pedestrian reference

This is the frozen ResNet50 accuracy-reference run for MobileADAS3D-S1. It fine-tunes the published MonoDETR checkpoint on the Chen 3,712-image train split after mapping Car/Van/Truck/Tram to the native Car ID and Pedestrian/Person_sitting to the native Pedestrian ID. Use a GPU runtime. Do not change parameters without assigning a new reference ID.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, shlex, shutil, subprocess, sys
MOBILE_REPO = Path('/content/mobile_adas3d')
MONODETR_REPO = Path('/content/MonoDETR')
MONODETR_COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT = Path('/content/kitti')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
MONODETR_KITTI = Path('/content/monodetr_kitti')
OFFICIAL_CHECKPOINT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/teachers/monodetr/checkpoint_best.pth')
OUTPUT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0')
RUN_NAME = 'monodetr_r0_vehicle_pedestrian'
MAX_EPOCHS = 195
SAVE_FREQUENCY = 5
BATCH_SIZE = 16
def run(command, cwd=None, env=None):
    command = [str(x) for x in command]; print('+', shlex.join(command), flush=True)
    merged = os.environ.copy(); merged.update(env or {})
    result = subprocess.run(command, cwd=cwd, env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])

In [ ]:
# Fetch pinned source and apply current-Colab plus product-taxonomy patches.
if not MOBILE_REPO.exists(): run(['git', 'clone', 'https://github.com/Ali-RT/mobile_adas3d.git', MOBILE_REPO])
else: run(['git', 'pull', '--ff-only'], cwd=MOBILE_REPO)
if not MONODETR_REPO.exists(): run(['git', 'clone', 'https://github.com/ZrrSkywalker/MonoDETR.git', MONODETR_REPO])
run(['git', 'fetch', '--all'], cwd=MONODETR_REPO)
run(['git', 'checkout', MONODETR_COMMIT], cwd=MONODETR_REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'pyyaml', 'scipy', 'opencv-python-headless', 'numba', 'scikit-image', 'tqdm', 'ninja'])
run([sys.executable, 'scripts/patch_monodetr_colab_compat.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
run([sys.executable, 'scripts/patch_monodetr_product_taxonomy.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
run([sys.executable, 'scripts/patch_monodetr_verbose_resume.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
run([sys.executable, 'scripts/patch_monodetr_checkpoint_metadata.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
ops = MONODETR_REPO / 'lib/models/monodetr/ops'
shutil.rmtree(ops / 'build', ignore_errors=True)
run([sys.executable, 'setup.py', 'build', 'install'], cwd=ops, env={'MAX_JOBS': '2'})
run([sys.executable, '-c', 'import torch, MultiScaleDeformableAttention; from lib.models.monodetr import build_monodetr; print(torch.__version__, torch.cuda.get_device_name(0))'], cwd=MONODETR_REPO)

In [ ]:
# Create a zero-copy KITTI view from local staging when available, otherwise Drive.
def resolve(root, names):
    for name in names:
        path = root / name
        if path.is_dir(): return path
sources = {}
for key, names in {'image_2':['training/image_2','training/image_02'], 'label_2':['training/label_2','training/label_02'], 'calib':['training/calib']}.items():
    sources[key] = resolve(LOCAL_DATASET_ROOT, names) or resolve(DRIVE_DATASET_ROOT, names)
if any(path is None for path in sources.values()): raise FileNotFoundError(f'Missing KITTI sources: {sources}')
(MONODETR_KITTI/'training').mkdir(parents=True, exist_ok=True)
(MONODETR_KITTI/'ImageSets').mkdir(parents=True, exist_ok=True)
for name, target in sources.items():
    link = MONODETR_KITTI/'training'/name
    if link.is_symlink() and link.resolve() == target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target, target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt', MONODETR_KITTI/'ImageSets'/f'{split}.txt')
assert len((MONODETR_KITTI/'ImageSets/train.txt').read_text().splitlines()) == 3712
assert len((MONODETR_KITTI/'ImageSets/val.txt').read_text().splitlines()) == 3769
print('KITTI R0 view:', MONODETR_KITTI, sources)

In [ ]:
# Build the immutable R0 initialization, resolved config, and hash manifest.
if not OFFICIAL_CHECKPOINT.is_file(): raise FileNotFoundError(OFFICIAL_CHECKPOINT)
run([sys.executable, 'scripts/prepare_monodetr_r0_reference.py', '--monodetr-repo', MONODETR_REPO, '--dataset-root', MONODETR_KITTI, '--official-checkpoint', OFFICIAL_CHECKPOINT, '--output-root', OUTPUT_ROOT, '--run-name', RUN_NAME, '--max-epochs', MAX_EPOCHS, '--save-frequency', SAVE_FREQUENCY, '--batch-size', BATCH_SIZE], cwd=MOBILE_REPO)
CONFIG = MONODETR_REPO/'configs/monodetr_r0_vehicle_pedestrian.yaml'
RUN_DIR = OUTPUT_ROOT/RUN_NAME
print(CONFIG.read_text())
print((RUN_DIR/'experiment_manifest.json').read_text())

## Real R0 training

The next cell is the full GPU training loop. Expect per-batch losses, epoch progress, validation every 5 epochs, and durable Drive checkpoints. Batch size 16 is the frozen setting. If it does not fit, stop and report the GPU and error before changing it.

In [ ]:
# Automatically resume the newest complete Drive checkpoint. A failed or partial file is skipped.
import re, torch, yaml
checkpoint_pattern = re.compile(r'^checkpoint_epoch_(\d+)\.pth$')
valid_checkpoints = []
for path in RUN_DIR.glob('checkpoint_epoch_*.pth'):
    match = checkpoint_pattern.match(path.name)
    if not match: continue
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False)
        epoch = int(payload.get('epoch', -1))
        if epoch != int(match.group(1)): raise ValueError(f'filename epoch {match.group(1)} != payload epoch {epoch}')
        if payload.get('model_state') is None: raise ValueError('model_state missing')
        if payload.get('optimizer_state') is None: raise ValueError('optimizer_state missing')
        valid_checkpoints.append((epoch, path))
        print(f'Valid resume checkpoint: epoch={epoch} size={path.stat().st_size/1e6:.1f}MB {path}')
    except Exception as error:
        print(f'Skipping invalid checkpoint {path}: {type(error).__name__}: {error}')
latest = max(valid_checkpoints, default=None, key=lambda item: item[0])
run_cfg = yaml.safe_load(CONFIG.read_text())
if latest is None:
    START_EPOCH = 0
    CONFIG_TO_RUN = CONFIG
    print('Starting fresh from the published MonoDETR R0 initialization.')
else:
    START_EPOCH, RESUME_CHECKPOINT = latest
    run_cfg['trainer'].pop('pretrain_model', None)
    run_cfg['trainer']['resume_model'] = str(RESUME_CHECKPOINT)
    run_cfg['trainer']['max_epoch'] = MAX_EPOCHS
    CONFIG_TO_RUN = MONODETR_REPO/'configs/monodetr_r0_vehicle_pedestrian_resume.yaml'
    CONFIG_TO_RUN.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
    print(f'Resuming R0 after completed epoch {START_EPOCH}: {RESUME_CHECKPOINT}')
print('Training config:', CONFIG_TO_RUN)
print(f'Remaining epochs: {max(0, MAX_EPOCHS - START_EPOCH)}')

In [ ]:
# Real training. Re-run the previous cell and this cell after any interruption.
if START_EPOCH >= MAX_EPOCHS:
    print(f'R0 already reached epoch {START_EPOCH}; no training required.')
else:
    run([sys.executable, '-u', 'tools/train_val.py', '--config', CONFIG_TO_RUN], cwd=MONODETR_REPO)

In [ ]:
# Durable artifacts. Product-taxonomy checkpoint sweep is the next gate.
print('Run directory:', RUN_DIR)
for path in sorted(RUN_DIR.glob('checkpoint*.pth')):
    print(path.name, round(path.stat().st_size / 1e6, 1), 'MB')
print('Do not select checkpoint_best.pth as final R0 yet: upstream selects Car AP only.')